# Heterogeneous Treatment Effects: Netflix Recommendation Algorithm

## Problem Statement

Netflix ran an A/B test comparing a **new recommendation algorithm** against the old one. The overall ATE shows a modest positive effect on watch time. But should they roll it out to everyone?

**The hidden problem**: The new algorithm disrupts veteran users' carefully curated recommendations, actually *reducing* their watch time. Meanwhile, new users benefit enormously because the new algorithm is better at cold-start recommendations.

**What we'll show**: Why reporting just the ATE is misleading, and how HTE methods reveal who benefits, who's harmed, and what the optimal rollout strategy is.

### Causal Question
$$CATE(x) = E[\text{WatchTime}(\text{new}) - \text{WatchTime}(\text{old}) \mid X = x]$$

Where $X$ includes user tenure, age, genre diversity, and viewing frequency.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

try:
    from econml.dml import CausalForestDML
    HAS_ECONML = True
    print("econml available — will use CausalForestDML")
except ImportError:
    HAS_ECONML = False
    print("econml not installed — using manual causal forest implementation")

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')
np.random.seed(42)

print("Setup complete.")

## Step 1: Simulate the A/B Test Data

We simulate 20,000 Netflix users randomly assigned to new vs old recommendation algorithm.

**Key design choice — heterogeneous treatment effects by tenure:**
- **New users** (tenure < 6 months): +2.0 hours/week (new algo solves cold-start problem)
- **Medium users** (6–24 months): +0.5 hours/week (modest improvement)
- **Veteran users** (24+ months): −0.3 hours/week (algo disrupts curated preferences)

This is realistic: recommendation algorithms often face a **personalization vs exploration** trade-off that affects user segments differently.

In [ ]:
n = 20000

tenure_months = np.random.exponential(scale=18, size=n).clip(0.5, 120)
age = np.random.normal(35, 12, size=n).clip(18, 75)
genre_diversity = np.random.beta(2, 5, size=n)  # 0-1, most users watch few genres
viewing_frequency = np.random.gamma(shape=3, scale=2, size=n).clip(0.5, 30)  # hours/week baseline

treatment = np.random.binomial(1, 0.5, size=n)  # random assignment

# True CATE as a function of tenure (the main driver)
# Smooth function: high for new users, drops through zero for veterans
true_cate = 2.5 * np.exp(-tenure_months / 8) - 0.3
# Add minor interaction: genre-diverse users benefit slightly more
true_cate += 0.5 * genre_diversity

# Baseline outcome (without treatment)
baseline_watch_time = (
    5.0
    + 0.05 * tenure_months  # veterans watch more
    + 0.02 * age
    + 3.0 * genre_diversity
    + 0.8 * viewing_frequency
)

noise = np.random.normal(0, 2.0, size=n)
watch_time = baseline_watch_time + treatment * true_cate + noise
watch_time = watch_time.clip(0, None)

df = pd.DataFrame({
    'tenure_months': tenure_months,
    'age': age,
    'genre_diversity': genre_diversity,
    'viewing_frequency': viewing_frequency,
    'treatment': treatment,
    'watch_time': watch_time,
    'true_cate': true_cate
})

df['tenure_group'] = pd.cut(
    df['tenure_months'],
    bins=[0, 6, 24, 200],
    labels=['New (<6mo)', 'Medium (6-24mo)', 'Veteran (24+mo)']
)

print(f"Dataset: {len(df):,} users")
print(f"Treatment/control split: {df['treatment'].mean():.1%} treated")
print(f"\nTenure distribution:")
print(df['tenure_group'].value_counts().sort_index())
print(f"\nTrue CATE summary:")
print(f"  Mean: {true_cate.mean():.3f}")
print(f"  Range: [{true_cate.min():.3f}, {true_cate.max():.3f}]")
print(f"  Users with positive CATE: {(true_cate > 0).mean():.1%}")
print(f"  Users with negative CATE: {(true_cate < 0).mean():.1%}")
df.head()

## Step 2: The Misleading ATE

Let's compute what a standard A/B test analysis would report: the overall Average Treatment Effect.

**The punchline**: ATE is positive, so a naive analysis says "roll out to everyone." But this *hides* the fact that veteran users are being harmed.

In [ ]:
treated = df[df['treatment'] == 1]['watch_time']
control = df[df['treatment'] == 0]['watch_time']

ate = treated.mean() - control.mean()
se = np.sqrt(treated.var() / len(treated) + control.var() / len(control))

print("=" * 60)
print("STANDARD A/B TEST RESULT")
print("=" * 60)
print(f"ATE: {ate:.3f} hours/week")
print(f"95% CI: [{ate - 1.96*se:.3f}, {ate + 1.96*se:.3f}]")
print(f"Statistically significant: {'Yes' if abs(ate/se) > 1.96 else 'No'}")
print(f"\nNaive conclusion: Roll out new algorithm to all users.")

print(f"\n{'=' * 60}")
print("BUT LOOK AT SUBGROUPS...")
print("=" * 60)

for group in ['New (<6mo)', 'Medium (6-24mo)', 'Veteran (24+mo)']:
    subset = df[df['tenure_group'] == group]
    t = subset[subset['treatment'] == 1]['watch_time']
    c = subset[subset['treatment'] == 0]['watch_time']
    diff = t.mean() - c.mean()
    sub_se = np.sqrt(t.var() / len(t) + c.var() / len(c))
    true_mean = subset['true_cate'].mean()
    print(f"\n  {group}:")
    print(f"    Estimated effect: {diff:+.3f} hrs/wk (SE: {sub_se:.3f})")
    print(f"    True mean CATE:   {true_mean:+.3f} hrs/wk")
    print(f"    N = {len(subset):,}")

print(f"\n{'=' * 60}")
print("The ATE of {:.3f} masks that veterans are HURT by the new algorithm.".format(ate))
print("A blanket rollout would degrade experience for your most loyal users.")
print("=" * 60)

## Step 3: Why HTE, Not Just ATE?

### Alternative Considered: Just Report the ATE

The ATE says "roll out everywhere" — but this is **actively harmful** because:

1. **Veteran users are hurt**: They have carefully curated recommendation profiles. The new algorithm disrupts these, reducing their watch time. These are your *most valuable* users.

2. **The positive ATE is driven by new users**: Who benefit from better cold-start recommendations. But the overall number hides the composition.

3. **Optimal policy ≠ uniform policy**: The best strategy is to deploy the new algorithm for new users and keep the old one for veterans.

### Why Not Simple Subgroup Analysis?

You *could* just split by tenure bins and compare means. Problems:
- **Arbitrary bin boundaries**: Why 6 months? Why not 4 or 8?
- **Misses interactions**: What about new users who are also high-frequency viewers?
- **Multiple comparisons**: Testing many subgroups inflates false positives
- **Can't handle continuous heterogeneity**: CATE varies smoothly, not in discrete jumps

HTE methods solve all of these by estimating CATE as a *function* of covariates.

## Step 4: Method Comparison — T-Learner vs S-Learner vs Causal Forest

| Method | Approach | Strengths | Weaknesses |
|--------|---------|-----------|------------|
| **T-learner** | Separate models for treated/control: CATE = μ₁(x) − μ₀(x) | Simple to implement; uses any ML model | Doesn't share information across arms; high variance when arms have different distributions |
| **S-learner** | Single model with treatment as a feature | Shares information; lower variance | Treatment feature competes with stronger predictors; may underestimate heterogeneity |
| **Causal Forest** | Directly optimizes splits for treatment effect heterogeneity | Purpose-built for CATE; honest inference; feature importance for heterogeneity | More complex; requires `econml` or `grf`; slower |

**Our plan**: Implement T-learner first (simple baseline), then Causal Forest (purpose-built), and compare both to the true CATE.

## Step 5: T-Learner Implementation

The T-learner fits **two separate models**:
- $\hat{\mu}_1(x)$: predict outcome for treated units
- $\hat{\mu}_0(x)$: predict outcome for control units

Then: $\widehat{CATE}(x) = \hat{\mu}_1(x) - \hat{\mu}_0(x)$

Simple and intuitive, but each model only sees half the data.

In [ ]:
features = ['tenure_months', 'age', 'genre_diversity', 'viewing_frequency']
X = df[features].values
Y = df['watch_time'].values
T = df['treatment'].values

# T-Learner: separate models for each treatment arm
model_treated = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42
)
model_control = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42
)

model_treated.fit(X[T == 1], Y[T == 1])
model_control.fit(X[T == 0], Y[T == 0])

# CATE = predicted outcome under treatment - predicted outcome under control
cate_t_learner = model_treated.predict(X) - model_control.predict(X)

# Evaluate against true CATE
mse_t = np.mean((cate_t_learner - df['true_cate'].values) ** 2)
corr_t = np.corrcoef(cate_t_learner, df['true_cate'].values)[0, 1]

print("T-Learner Results")
print("=" * 40)
print(f"MSE vs true CATE:  {mse_t:.4f}")
print(f"Correlation:       {corr_t:.4f}")
print(f"Mean est. CATE:    {cate_t_learner.mean():.3f}")
print(f"Mean true CATE:    {df['true_cate'].mean():.3f}")

## Step 6: Causal Forest Implementation

Causal forests differ from standard random forests: each tree split is chosen to **maximize the difference in treatment effects** between the two child nodes, not to minimize prediction error.

If `econml` is available, we use `CausalForestDML` (doubly-robust, handles confounding). Otherwise, we implement a manual version using the R-learner idea:

1. Residualize outcome: $\tilde{Y} = Y - \hat{E}[Y|X]$
2. Residualize treatment: $\tilde{T} = T - \hat{E}[T|X]$ (= T − 0.5 in an RCT)
3. Regress $\tilde{Y}$ on $\tilde{T}$ using a forest, where the "pseudo-outcome" $\tilde{Y}/\tilde{T}$ approximates CATE

In [ ]:
if HAS_ECONML:
    cf = CausalForestDML(
        model_y=GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42),
        model_t=GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42),
        n_estimators=200,
        min_samples_leaf=20,
        random_state=42
    )
    cf.fit(Y, T, X=X)
    cate_forest = cf.effect(X).flatten()
    print("Used econml CausalForestDML")
else:
    # Manual causal forest via R-learner approach
    # Step 1: Residualize Y — remove effect of X on outcome
    y_model = GradientBoostingRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42
    )
    y_hat = cross_val_predict(y_model, X, Y, cv=5)
    y_resid = Y - y_hat

    # Step 2: Residualize T (in RCT, propensity = 0.5, so T_resid = T - 0.5)
    t_resid = T - 0.5

    # Step 3: Pseudo-outcome for CATE estimation
    # In the R-learner, CATE(x) solves: min E[(Y_resid - tau(X)*T_resid)^2]
    # Equivalent to regressing Y_resid/T_resid on X, weighted by T_resid^2
    pseudo_outcome = y_resid / t_resid  # = Y_resid / (T - 0.5)
    weights = t_resid ** 2

    cate_model = RandomForestRegressor(
        n_estimators=500, max_depth=8, min_samples_leaf=50, random_state=42
    )
    cate_model.fit(X, pseudo_outcome, sample_weight=weights)
    cate_forest = cate_model.predict(X)
    print("Used manual R-learner with RandomForest")

mse_cf = np.mean((cate_forest - df['true_cate'].values) ** 2)
corr_cf = np.corrcoef(cate_forest, df['true_cate'].values)[0, 1]

print(f"\nCausal Forest Results")
print("=" * 40)
print(f"MSE vs true CATE:  {mse_cf:.4f}")
print(f"Correlation:       {corr_cf:.4f}")
print(f"Mean est. CATE:    {cate_forest.mean():.3f}")
print(f"Mean true CATE:    {df['true_cate'].mean():.3f}")

print(f"\n{'=' * 50}")
print("Method Comparison")
print("=" * 50)
print(f"{'Method':<20} {'MSE':>10} {'Correlation':>12}")
print(f"{'-'*20} {'-'*10} {'-'*12}")
print(f"{'T-Learner':<20} {mse_t:>10.4f} {corr_t:>12.4f}")
print(f"{'Causal Forest':<20} {mse_cf:>10.4f} {corr_cf:>12.4f}")

## Step 7: Visualize CATE by Tenure

Tenure is the main driver of heterogeneity. Let's visualize how the estimated CATE varies with tenure and compare to the true CATE curve.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sort_idx = np.argsort(df['tenure_months'].values)
tenure_sorted = df['tenure_months'].values[sort_idx]
true_sorted = df['true_cate'].values[sort_idx]
tlearner_sorted = cate_t_learner[sort_idx]
forest_sorted = cate_forest[sort_idx]

# Panel 1: True CATE vs T-Learner
ax = axes[0]
ax.scatter(df['tenure_months'], cate_t_learner, alpha=0.03, s=5, color='steelblue', label='T-Learner estimates')
ax.plot(tenure_sorted, true_sorted, color='red', linewidth=2, label='True CATE')
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('CATE (hours/week)', fontsize=12)
ax.set_title('T-Learner vs True CATE', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(-3, 5)

# Panel 2: True CATE vs Causal Forest
ax = axes[1]
ax.scatter(df['tenure_months'], cate_forest, alpha=0.03, s=5, color='forestgreen', label='Causal Forest estimates')
ax.plot(tenure_sorted, true_sorted, color='red', linewidth=2, label='True CATE')
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('CATE (hours/week)', fontsize=12)
ax.set_title('Causal Forest vs True CATE', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(-3, 5)

# Panel 3: Binned comparison
ax = axes[2]
bins = np.arange(0, df['tenure_months'].max() + 5, 5)
df['tenure_bin'] = pd.cut(df['tenure_months'], bins=bins)
df['cate_tlearner'] = cate_t_learner
df['cate_forest'] = cate_forest

binned = df.groupby('tenure_bin', observed=True).agg(
    true_cate=('true_cate', 'mean'),
    tlearner=('cate_tlearner', 'mean'),
    forest=('cate_forest', 'mean'),
    tenure_mid=('tenure_months', 'mean')
).dropna()

ax.plot(binned['tenure_mid'], binned['true_cate'], 'r-o', linewidth=2, markersize=4, label='True CATE')
ax.plot(binned['tenure_mid'], binned['tlearner'], 'b--s', linewidth=1.5, markersize=4, label='T-Learner')
ax.plot(binned['tenure_mid'], binned['forest'], 'g--^', linewidth=1.5, markersize=4, label='Causal Forest')
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax.axhline(y=ate, color='orange', linestyle=':', alpha=0.7, label=f'Overall ATE ({ate:.2f})')
ax.fill_between(binned['tenure_mid'], 0, binned['true_cate'],
                where=binned['true_cate'] > 0, alpha=0.1, color='green', label='Positive effect zone')
ax.fill_between(binned['tenure_mid'], 0, binned['true_cate'],
                where=binned['true_cate'] < 0, alpha=0.1, color='red', label='Negative effect zone')
ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('CATE (hours/week)', fontsize=12)
ax.set_title('Binned CATE Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('cate_by_tenure.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey insight: Both methods correctly identify that CATE decreases with tenure")
print("and turns negative for veteran users — something the ATE completely hides.")

## Step 8: Feature Importance for Heterogeneity

Which features drive the most **variation** in CATE? This tells us what dimensions of heterogeneity matter for targeting.

We use permutation importance on the CATE model: shuffle each feature and measure how much the CATE predictions change.

In [ ]:
# Feature importance for treatment effect heterogeneity
# Approach: train a model to predict CATE from X, then measure feature importance

cate_predictor = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
cate_predictor.fit(X, cate_forest)  # predict estimated CATE from features

# Built-in importance (Gini-based)
gini_importance = pd.Series(
    cate_predictor.feature_importances_,
    index=features
).sort_values(ascending=True)

# Permutation importance (more reliable)
perm_imp = permutation_importance(
    cate_predictor, X, cate_forest,
    n_repeats=20, random_state=42
)
perm_importance = pd.Series(
    perm_imp.importances_mean,
    index=features
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = ['#2196F3' if v == gini_importance.max() else '#90CAF9' for v in gini_importance.values]
ax.barh(gini_importance.index, gini_importance.values, color=colors)
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Gini Importance for CATE Heterogeneity', fontsize=13, fontweight='bold')

ax = axes[1]
colors = ['#4CAF50' if v == perm_importance.max() else '#A5D6A7' for v in perm_importance.values]
ax.barh(perm_importance.index, perm_importance.values, color=colors,
        xerr=perm_imp.importances_std[perm_importance.index.map({f: i for i, f in enumerate(features)})],
        capsize=3)
ax.set_xlabel('Importance (R² decrease)', fontsize=12)
ax.set_title('Permutation Importance for CATE Heterogeneity', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('feature_importance_hte.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nHeterogeneity drivers (permutation importance):")
for feat in perm_importance.index[::-1]:
    print(f"  {feat:<25s} {perm_importance[feat]:.4f}")
print("\nAs expected, tenure_months dominates — it's the primary source of")
print("treatment effect heterogeneity. genre_diversity is a secondary driver.")

## Step 9: Policy Implications — Targeted vs Blanket Rollout

Now the actionable part: using CATE estimates to make a **better rollout decision**.

Three strategies:
1. **Status quo**: Keep old algorithm for everyone
2. **Blanket rollout**: Deploy new algorithm to everyone (what ATE suggests)
3. **Targeted rollout**: Deploy only to users with positive estimated CATE

In [ ]:
# Policy comparison
df['est_cate'] = cate_forest

# Strategy 1: Status quo (no one gets new algorithm)
value_status_quo = 0.0  # baseline

# Strategy 2: Blanket rollout (everyone gets new algorithm)
value_blanket = df['true_cate'].mean()  # realized if we deploy to all

# Strategy 3: Targeted rollout (deploy to est_cate > 0 only)
targeted_mask = df['est_cate'] > 0
value_targeted = df.loc[targeted_mask, 'true_cate'].sum() / len(df)

# Strategy 3b: Targeted with higher threshold for safety
safe_mask = df['est_cate'] > 0.2
value_safe = df.loc[safe_mask, 'true_cate'].sum() / len(df)

print("=" * 65)
print("POLICY COMPARISON: Expected Watch Time Gain (hours/user/week)")
print("=" * 65)
strategies = [
    ('Status quo (old algo for all)', value_status_quo, 0),
    ('Blanket rollout (new algo for all)', value_blanket, len(df)),
    ('Targeted (est. CATE > 0)', value_targeted, targeted_mask.sum()),
    ('Conservative (est. CATE > 0.2)', value_safe, safe_mask.sum()),
]

for name, value, n_treated in strategies:
    pct = n_treated / len(df) * 100
    print(f"\n  {name}")
    print(f"    Expected gain:  {value:+.4f} hrs/user/week")
    print(f"    Users treated:  {n_treated:,} ({pct:.1f}%)")

improvement = ((value_targeted - value_blanket) / value_blanket * 100)
print(f"\n{'=' * 65}")
print(f"Targeting improves value by {improvement:.1f}% over blanket rollout")
print(f"while protecting {(~targeted_mask).sum():,} users from a negative experience.")
print("=" * 65)

# Visualize the targeting decision
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: CATE distribution with targeting threshold
ax = axes[0]
ax.hist(df['est_cate'], bins=60, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(x=0, color='red', linewidth=2, linestyle='--', label='Targeting threshold')
ax.axvline(x=ate, color='orange', linewidth=2, linestyle=':', label=f'Overall ATE ({ate:.2f})')
n_negative = (df['est_cate'] < 0).sum()
n_positive = (df['est_cate'] >= 0).sum()
ax.text(0.02, 0.95, f'Deploy new algo\n({n_positive:,} users)',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
ax.text(0.02, 0.75, f'Keep old algo\n({n_negative:,} users)',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightsalmon', alpha=0.5))
ax.set_xlabel('Estimated CATE (hours/week)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('CATE Distribution with Targeting Decision', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# Panel 2: Targeting decision by tenure
ax = axes[1]
deploy = df[df['est_cate'] >= 0]
keep_old = df[df['est_cate'] < 0]
ax.scatter(deploy['tenure_months'], deploy['est_cate'], alpha=0.05, s=5,
           color='green', label=f'Deploy new ({len(deploy):,})')
ax.scatter(keep_old['tenure_months'], keep_old['est_cate'], alpha=0.05, s=5,
           color='red', label=f'Keep old ({len(keep_old):,})')
ax.axhline(y=0, color='black', linewidth=1.5, linestyle='--')
ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('Estimated CATE', fontsize=12)
ax.set_title('Targeting Decision by User Tenure', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, markerscale=10)

plt.tight_layout()
plt.savefig('targeting_policy.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

### 1. ATE Can Be Actively Misleading
The overall ATE was positive, suggesting a blanket rollout. But this *hides* that the new algorithm hurts veteran users — your most valuable cohort. Reporting only the ATE would have led to a harmful decision.

### 2. HTE Methods Reveal Actionable Heterogeneity
Both the T-learner and Causal Forest correctly identified that:
- New users benefit greatly (CATE ≈ +2 hrs/wk)
- Veterans are harmed (CATE ≈ −0.3 hrs/wk)
- Tenure is the dominant driver of heterogeneity

### 3. Targeting Beats Blanket Rollout
Deploying the new algorithm *only* to users with positive estimated CATE improves the overall value while protecting veterans from degraded experience. This is the core promise of personalized treatment assignment.

### 4. Method Choice Matters
- **T-learner**: Good baseline, easy to implement. Use when you want quick estimates.
- **Causal Forest**: Purpose-built for heterogeneity discovery. Better when you don't know which features drive HTE.
- Both benefit from the RCT setting (no confounding to worry about).

### 5. Connection to Uplift Modeling
This is exactly the same as uplift modeling in marketing:
- **Persuadables** (positive CATE): Deploy new algorithm
- **Sleeping dogs** (negative CATE): Keep old algorithm
- The CATE estimate *is* the uplift score

### Next Steps in Practice
- Validate CATE estimates on held-out data or a follow-up experiment
- Use CATE confidence intervals to be conservative with the targeting threshold
- Monitor for concept drift: user segments shift over time
- Consider fairness: does targeting create disparate impact across demographic groups?